## Sommaire
- [Fonctions présentes dans Fonction](#fonctions-présentes-dans-fonction)
- [Classe Compte](#classe-compte)
  - [Paramètres](#paramètres)
  - [Fonctions](#fonctions)
- [Classe Comptabilité](#classe-comptabilité)
  - [Paramètres](#paramètres-1)
  - [Fonctions](#fonctions-1)

# Import

In [18]:
# Standard library
import os
import shutil

# Third-party
import pandas as pd
import openpyxl
from tkcalendar import DateEntry

# Tkinter
import tkinter as tk
from tkinter import ttk, messagebox, filedialog, simpledialog

# matplotlib
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import matplotlib.pyplot as plt

# Local modules
from Fonction import *
from Classe import *


# Documentation

## Fonctions présentes dans `Fonction`

| Nom de la fonction | Paramètres | Description |
|--------------------|------------|-------------|
| `filtrer_par_date` | tab, date_debut, date_fin | Filtrer par plage de dates |
| `filtrer_par_mois` | tab, mois_debut, mois_fin | Filtrer par plage de mois |
| `filtrer_par_annee` | tab, annee_debut, annee_fin | Filtrer par plage d’années |
| `filtrer` | tab1, colonne, valeur | Filtrer par colonne et valeur |
| `trier_tableau` | tab, colonnes, croissant=True | Trier un tableau selon des colonnes |
| `Cummuler_Name` | tab, colonne=None, valeurs=None | Cumul par nom |
| `supprimer_entree` | tab, index | Supprimer une entrée du tableau |
| `ajouter_entree` | tab, date, intitule, categorie, classe, type_, valeur | Ajouter une entrée dans le tableau |

---

## Classe `Compte`

### Paramètres
| Nom | Description |
|-----|-------------|
| `chemin` | chemin du compte |
| `chemin_previsonnel` | lié dans `./prev/` dans le chemin |
| `name` | nom du compte |
| `state` | état du compte (`open` / `close` / `blocked`) |
| `back_up` | nombre de back_up |
| `compte_actif` | dataframe pour le compte actif |
| `compte_previsionnel` | dataframe pour le compte prévisionnel |

### Fonctions
| Nom de la fonction | Paramètres | Description |
|--------------------|------------|-------------|
| `__init__` | chemin_, state_="open", back_up_=0 | Initialiser un compte |
| `load_compte` | self | Charger le compte |
| `save_compte_actif` | self | Sauvegarder le compte actif |
| `save_previsionnel` | self | Sauvegarder le prévisionnel |
| `__repr__` | self | Représentation textuelle du compte |

---

## Classe `Comptabilité`

### Paramètres
| Nom | Description |
|-----|-------------|
| `chemin_txt` | chemin du fichier txt qui stocke tous les comptes |
| `comptes` | liste d’objets `Compte` |
| `erreur` | liste des erreurs |

### Fonctions
| Nom de la fonction | Paramètres | Description |
|--------------------|------------|-------------|
| `__init__` | chemin_txt | Initialiser la comptabilité |
| `_load_comptes` | self | Charger les comptes |
| `ajouter_compte` | chemin_compte, etat="open" | Ajouter un compte |
| `supprimer_compte` | identifiant, supprimer_fichier=True | Supprimer un compte |
| `changer_etat` | nom_compte, nouvel_etat | Changer l’état d’un compte |
| `__repr__` | self | Représentation textuelle de la comptabilité |
| `lister_comptes` | self | Lister les comptes |
| `lister_erreurs` | self | Lister les erreurs |


## Fonction pour l'interface graphique 

In [5]:
# -------------------------
# Combobox avec autocomplétion
# -------------------------
class AutocompleteCombobox(ttk.Combobox):
    def set_completion_list(self, completion_list):
        """Charge une liste de valeurs uniques (insensible à la casse)"""
        clean_list = [str(x) for x in completion_list if pd.notna(x)]
        self._completion_list = sorted(set(clean_list), key=lambda v: v.lower())
        self._filtered_list = self._completion_list
        self["values"] = self._completion_list
        self.bind("<KeyRelease>", self._on_keyrelease)

    def _on_keyrelease(self, event):
        """Filtrer dynamiquement, insensible à la casse"""
        text = self.get().lower()
        if text == "":
            self._filtered_list = self._completion_list
        else:
            self._filtered_list = [v for v in self._completion_list if text in v.lower()]
        self["values"] = self._filtered_list


In [7]:
class Header(tk.Frame):
    def __init__(self, parent, shared_data):
        super().__init__(parent, bg="lightblue", height=50)
        self.shared_data = shared_data

        tk.Label(self, text="Menu :", bg="lightblue", font=("Arial", 12)).pack(side="left", padx=10)

        # Variable de choix
        self.shared_data["choix"] = tk.StringVar(value="Sélectionner un compte")

        # Variables pour les labels
        self.revenu_var = tk.StringVar(value="Revenu : 0")
        self.depense_var = tk.StringVar(value="Dépense : 0")
        self.ecart_var = tk.StringVar(value="Écart : 0")

        # Placeholders (seront recréés dans refresh_menu)
        self.menu = None
        self.refresh_menu()

        # Labels dynamiques
        tk.Label(self, textvariable=self.ecart_var, bg="lightblue", font=("Arial", 12)).pack(side="right", padx=10)
        tk.Label(self, textvariable=self.depense_var, bg="lightblue", font=("Arial", 12)).pack(side="right", padx=10)
        tk.Label(self, textvariable=self.revenu_var, bg="lightblue", font=("Arial", 12)).pack(side="right", padx=10)

    def refresh_menu(self):
        """Reconstruit le menu déroulant avec les comptes actuels"""
        if self.menu:
            self.menu.destroy()

        options = [compte.name for compte in self.shared_data["Comptabilite"].comptes]
        self.menu = tk.OptionMenu(self, self.shared_data["choix"], *options, command=lambda _: self.MAJ())
        self.menu.pack(side="left", padx=10)

    def MAJ(self):
        """Met à jour les infos de revenu, dépense, écart"""
        nom_compte = self.shared_data["choix"].get()
        if not nom_compte or nom_compte == "Sélectionner un compte":
            return

        compte = next((c for c in self.shared_data["Comptabilite"].comptes if c.name == nom_compte), None)
        if compte is None:
            return

        self.shared_data["Name"].set(compte.name)
        self.shared_data["Compte"] = compte.compte_actif

        Cumulle = Cummuler_Name(compte.compte_actif)

        # ✅ corriger les guillemets
        self.revenu_var.set(f"Revenu : {Cumulle['Revenu'][0]}")
        self.depense_var.set(f"Dépense : {Cumulle['Depense'][0]}")
        self.ecart_var.set(f"Écart : {Cumulle['Ecart'][0]}")


In [9]:
class Gestion_Compte(ttk.Frame):
    def __init__(self, parent, shared_data):
        super().__init__(parent)
        self.compta = shared_data["Comptabilite"]
        self.name = shared_data["Name"]  # StringVar partagé

        # Style pour gros boutons
        style = ttk.Style()
        style.configure(
            "Big.TButton",
            font=("Arial", 14, "bold"),
            padding=(20, 10)
        )

        self._build_ui()
        self._refresh_tableau()

    def _build_ui(self):
        colonne = ["Nom", "Etat du Compte", "Argent"]
        ancre = [tk.W, tk.CENTER, tk.E]
        
        self.tree = ttk.Treeview(
            self, columns=colonne, show="headings", height=12
        )
        for i, col in enumerate(colonne):
            self.tree.heading(col, text=col)
            self.tree.column(col, width=250, anchor=ancre[i])
        self.tree.pack(fill=tk.BOTH, expand=True, pady=10, padx=10)

        # Met à jour `Name` quand on clique sur une ligne
        self.tree.bind("<<TreeviewSelect>>", self._on_tree_select)

        frame_btn = tk.Frame(self)
        frame_btn.pack(pady=15)

        # Bouton Ajouter (menu)
        btn_ajouter = ttk.Button(frame_btn, text="Ajouter", style="Big.TButton")
        btn_ajouter.grid(row=0, column=0, padx=10)
        btn_ajouter.bind("<Button-1>", lambda e: self._menu_ajouter(btn_ajouter))

        # Bouton Supprimer
        ttk.Button(frame_btn, text="Supprimer", style="Big.TButton", command=self._supprimer).grid(row=0, column=1, padx=10)

        # Bouton Rafraîchir
        ttk.Button(frame_btn, text="Rafraîchir", style="Big.TButton", command=self._refresh_tableau).grid(row=0, column=2, padx=10)

    def _on_tree_select(self, event):
        """Met à jour Name quand on sélectionne une ligne dans le tableau."""
        sel = self.tree.selection()
        if sel:
            nom = sel[0]   # on a mis iid=compte.name
            self.name.set(f"Compte sélectionné : {nom}")

    def _refresh_tableau(self):
        """Recharge le tableau avec les comptes actuels"""
        for row in self.tree.get_children():
            self.tree.delete(row)

        for compte in self.compta.comptes:
            try:
                argent = Cummuler_Name(compte.compte_actif)["Ecart"][0]
            except Exception:
                argent = "N/A"
            # iid = nom du compte → pratique pour retrouver la sélection
            self.tree.insert("", tk.END, iid=compte.name,
                             values=(compte.name, compte.state, argent))

    def _supprimer(self): 
        """Supprime le compte sélectionné (tableau ou header)."""
        # 1) priorité au tableau
        sel = self.tree.selection()
        if sel:
            nom = sel[0]
        else:
            # 2) sinon via le header
            nom = self.name.get().replace("Compte sélectionné : ", "").strip()

        if not nom:
            messagebox.showwarning("Aucun compte", "Sélectionnez un compte dans la liste ou via le menu.")
            return

        if not any(c.name == nom for c in self.compta.comptes):
            messagebox.showerror("Introuvable", f"Le compte '{nom}' n'existe plus.")
            self._refresh_tableau()
            return

        if not messagebox.askyesno("Confirmer", f"Supprimer le compte '{nom}' ?"):
            return

        try:
            self.compta.supprimer_compte(nom, supprimer_fichier=True)
        except Exception as e:
            messagebox.showerror("Erreur", str(e))
            return

        # Mise à jour UI
        if nom in self.tree.get_children():
            self.tree.delete(nom)
        self.name.set("Compte sélectionné : ")
        self._refresh_tableau()

        # Si header a une méthode pour MAJ menu
        if hasattr(self.master, "header") and hasattr(self.master.header, "refresh_menu"):
            try:
                self.master.header.refresh_menu()
            except Exception:
                pass
        self.master.refresh_all() 

    def _menu_ajouter(self, bouton):
        """Menu déroulant pour ajouter un compte"""
        menu = tk.Menu(self, tearoff=0)
        menu.add_command(label="Créer un fichier Excel", command=self._ajouter_creer_fichier)
        menu.add_command(label="Référencer un Excel existant", command=self._ajouter_excel_existant)

        x = bouton.winfo_rootx()
        y = bouton.winfo_rooty() + bouton.winfo_height()
        menu.post(x, y)
        

    def _ajouter_creer_fichier(self):
        """Créer un nouveau fichier Excel et l'ajouter dans Comptabilité"""
        nom_compte = simpledialog.askstring("Nouveau compte", "Entrez le nom du compte :")
        if not nom_compte:
            return  # Annulé

        try:
            dossier = os.path.dirname(self.compta.chemin_txt)
            fichier = os.path.join(dossier, f"{nom_compte}.xlsx")

            if os.path.exists(fichier):
                messagebox.showerror("Erreur", f"Le fichier '{nom_compte}.xlsx' existe déjà.")
                return

            # Créer un Excel avec en-tête
            wb = openpyxl.Workbook()
            ws = wb.active
            ws.title = "Compte"
            ws.append(["Date", "Intitulé", "Catégorie", "Classe", "Type", "Valeur"])
            wb.save(fichier)

            # 👉 Laisser Comptabilité gérer l'ajout dans le TXT
            self.compta.ajouter_compte(fichier, etat="open")

            self._refresh_tableau()
            messagebox.showinfo("Succès", f"Compte '{nom_compte}' créé et ajouté.")

        except Exception as e:
            messagebox.showerror("Erreur", str(e))
        self.master.refresh_all() 
    def _ajouter_excel_existant(self):
        """Référencer un Excel existant et l'ajouter à Comptabilité"""
        fichier = filedialog.askopenfilename(
            title="Choisir un fichier Excel existant",
            filetypes=[("Fichier Excel", "*.xlsx *.xls")]
        )
        if fichier:
            try:
                # 👉 Laisser Comptabilité gérer le .txt
                self.compta.ajouter_compte(fichier, etat="open")

                self._refresh_tableau()
                messagebox.showinfo("Succès", f"Compte référencé : {os.path.basename(fichier)}")
            except Exception as e:
                messagebox.showerror("Erreur", str(e))


In [22]:


class Tableau(ttk.Frame):
    def __init__(self, parent, shared_data):
        self.compte = shared_data["Compte"]
        self._build_ui()
    def _build_ui(self) : 


        colonne = ["Nom", "Etat du Compte", "Argent"]
        ancre = [tk.W, tk.CENTER, tk.E]
        
        self.tree = ttk.Treeview(
            self, columns=colonne, show="headings", height=12
        )
        for i, col in enumerate(colonne):
            self.tree.heading(col, text=col)
            self.tree.column(col, width=250, anchor=ancre[i])
        self.tree.pack(fill=tk.BOTH, expand=True, pady=10, padx=10)

        # Met à jour `Name` quand on clique sur une ligne
        self.tree.bind("<<TreeviewSelect>>", self._on_tree_select)

        frame_btn = tk.Frame(self)
        frame_btn.pack(pady=15)

        # Bouton Ajouter (menu)
        btn_ajouter = ttk.Button(frame_btn, text="Ajouter", style="Big.TButton")
        btn_ajouter.grid(row=0, column=0, padx=10)
        btn_ajouter.bind("<Button-1>", lambda e: self._menu_ajouter(btn_ajouter))

        # Bouton Supprimer
        ttk.Button(frame_btn, text="Supprimer", style="Big.TButton", command=self._supprimer).grid(row=0, column=1, padx=10)

        # Bouton Rafraîchir
        ttk.Button(frame_btn, text="Rafraîchir", style="Big.TButton", command=self._refresh_tableau).grid(row=0, column=2, padx=10)




In [24]:
class Main_Interface(tk.Tk):
    def __init__(self, chemin_txt):
        super().__init__()
        self.title("Interface de gestion d'argent")
        self.geometry("1000x900")

        self.compta = Comptabilité(chemin_txt)

        # Données partagées
        self.shared_data = {
            "Comptabilite": self.compta,
            "Name": tk.StringVar(value="Aucun compte sélectionné"),
            "Compte": None
        }

        # Barre du haut
        self.header = Header(self, self.shared_data)
        self.header.pack(side="top", fill="x")

        notebook = ttk.Notebook(self)
        notebook.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)

        

        # 👇 Astuce : on donne self (Main_Interface) comme parent
        # puis on ajoute le frame au notebook
        self.Gestion_Compte_onglet = Gestion_Compte(self, self.shared_data)
        notebook.add(self.Gestion_Compte_onglet, text="Comptes")

        self.Gestion_Compte_onglet = Tableau(self, self.shared_data)
        notebook.add(self.Gestion_Compte_onglet, text="Gestion")
        
    def refresh_all(self):
        self.header.refresh_menu()  # 🔥 le menu déroulant se reconstruit
        self.header.MAJ()           # 🔥 les labels se mettent à jour
        self.Gestion_Compte_onglet._refresh_tableau()





if __name__ == "__main__":
    app = Main_Interface("D:/Fichier/GitHub/Python_Budget/Notebook/CSV/liste.txt")
    app.mainloop()


AttributeError: 'Tableau' object has no attribute 'tk'